# ImageNet Object Localization Challenge Dataset Explorer 🔍

## Dataset Overview
- **Source**: Kaggle ImageNet Object Localization Challenge
- **Location**: `/home/ubuntu/Downloads/`
- **Classes**: 1000 object categories
- **Task**: Object classification and localization

## Dataset Structure (ILSVRC Format)
```
ILSVRC/
├── Data/
│   └── CLS-LOC/
│       ├── train/           # Training images (1000 class folders)
│       ├── val/             # Validation images (50,000 images)
│       └── test/            # Test images (100,000 images)
├── Annotations/
│   └── CLS-LOC/
│       ├── train/           # Training bounding box annotations (XML)
│       └── val/             # Validation bounding box annotations (XML)
└── ImageSets/
    └── CLS-LOC/
        ├── train_cls.txt    # Training image list
        ├── val.txt          # Validation image list
        └── test.txt         # Test image list
```

This notebook provides step-by-step exploration of the ImageNet dataset.

In [ ]:
# Step 1: Import Required Libraries
import os
import glob
import random
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from tqdm import tqdm

# Set plotting style
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style("whitegrid")

print("✅ Libraries imported successfully!")

In [ ]:
# Step 2: Set Dataset Paths
# Update this path to where you extracted the ImageNet dataset
DATASET_ROOT = "/home/ubuntu/Downloads/ILSVRC"  # Main extracted folder

# Define key directories
DATA_DIR = os.path.join(DATASET_ROOT, "Data", "CLS-LOC")
ANNOTATIONS_DIR = os.path.join(DATASET_ROOT, "Annotations", "CLS-LOC")
IMAGESETS_DIR = os.path.join(DATASET_ROOT, "ImageSets", "CLS-LOC")

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

TRAIN_ANNOTATIONS_DIR = os.path.join(ANNOTATIONS_DIR, "train")
VAL_ANNOTATIONS_DIR = os.path.join(ANNOTATIONS_DIR, "val")

print(f"📁 Dataset root: {DATASET_ROOT}")
print(f"📁 Data directory: {DATA_DIR}")
print(f"📁 Annotations directory: {ANNOTATIONS_DIR}")
print(f"📁 ImageSets directory: {IMAGESETS_DIR}")

# Check if directories exist
directories = {
    "Dataset Root": DATASET_ROOT,
    "Train Images": TRAIN_DIR,
    "Val Images": VAL_DIR,
    "Test Images": TEST_DIR,
    "Train Annotations": TRAIN_ANNOTATIONS_DIR,
    "Val Annotations": VAL_ANNOTATIONS_DIR,
    "ImageSets": IMAGESETS_DIR
}

for name, path in directories.items():
    if os.path.exists(path):
        print(f"✅ {name}: Found")
    else:
        print(f"❌ {name}: Not found at {path}")

In [ ]:
# Step 3: Load ImageNet Class Information
# ImageNet classes follow WordNet synset IDs (e.g., n01440764)

def load_class_names():
    """Load ImageNet class names from training directory structure"""
    class_info = {}
    
    if os.path.exists(TRAIN_DIR):
        class_dirs = [d for d in os.listdir(TRAIN_DIR) 
                     if os.path.isdir(os.path.join(TRAIN_DIR, d))]
        
        for i, class_dir in enumerate(sorted(class_dirs)):
            class_info[class_dir] = {
                'synset_id': class_dir,
                'class_index': i,
                'class_name': class_dir  # Will be synset ID if no mapping available
            }
    
    return class_info

# Load class information
class_info = load_class_names()
print(f"📊 Found {len(class_info)} classes")

# Display first 10 classes
if class_info:
    print("\n🔍 First 10 classes:")
    for i, (synset, info) in enumerate(list(class_info.items())[:10]):
        print(f"  {i+1:3d}. {synset}")
else:
    print("⚠️ No classes found. Check if the dataset is properly extracted.")

In [ ]:
# Step 4: Analyze Dataset Structure and Statistics

def analyze_dataset_structure():
    """Analyze the structure and get basic statistics"""
    stats = {
        'train': {'classes': 0, 'images': 0, 'total_size_gb': 0},
        'val': {'classes': 0, 'images': 0, 'total_size_gb': 0},
        'test': {'classes': 0, 'images': 0, 'total_size_gb': 0}
    }
    
    # Analyze training set
    if os.path.exists(TRAIN_DIR):
        train_classes = [d for d in os.listdir(TRAIN_DIR) 
                        if os.path.isdir(os.path.join(TRAIN_DIR, d))]
        stats['train']['classes'] = len(train_classes)
        
        # Count images per class (sample first 10 classes for speed)
        total_train_images = 0
        sample_classes = train_classes[:10] if len(train_classes) > 10 else train_classes
        
        for class_dir in sample_classes:
            class_path = os.path.join(TRAIN_DIR, class_dir)
            images = [f for f in os.listdir(class_path) 
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            total_train_images += len(images)
        
        # Estimate total images (extrapolate from sample)
        if sample_classes:
            avg_per_class = total_train_images / len(sample_classes)
            stats['train']['images'] = int(avg_per_class * len(train_classes))
    
    # Analyze validation set
    if os.path.exists(VAL_DIR):
        val_images = [f for f in os.listdir(VAL_DIR) 
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        stats['val']['images'] = len(val_images)
        stats['val']['classes'] = 1000  # Val images are in single directory
    
    # Analyze test set
    if os.path.exists(TEST_DIR):
        test_images = [f for f in os.listdir(TEST_DIR) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        stats['test']['images'] = len(test_images)
        stats['test']['classes'] = 1000  # Test images are in single directory
    
    return stats

# Get dataset statistics
print("🔍 Analyzing dataset structure...")
dataset_stats = analyze_dataset_structure()

# Display statistics
print("\n📊 Dataset Statistics:")
print("=" * 50)
for split, stats in dataset_stats.items():
    print(f"{split.upper():>8}: {stats['classes']:,} classes, {stats['images']:,} images")

# Total images
total_images = sum(stats['images'] for stats in dataset_stats.values())
print(f"{'TOTAL':>8}: {total_images:,} images")

In [ ]:
# Step 5: Explore Training Set - Images per Class Distribution

def analyze_training_distribution():
    """Analyze the distribution of images per class in training set"""
    if not os.path.exists(TRAIN_DIR):
        print("❌ Training directory not found")
        return None
    
    class_counts = []
    class_names = []
    
    # Get list of class directories
    train_classes = [d for d in os.listdir(TRAIN_DIR) 
                    if os.path.isdir(os.path.join(TRAIN_DIR, d))]
    
    print(f"📊 Analyzing {len(train_classes)} training classes...")
    
    # Sample classes for faster analysis (or use all if small dataset)
    sample_size = min(50, len(train_classes))  # Analyze first 50 classes
    sample_classes = train_classes[:sample_size]
    
    for class_dir in tqdm(sample_classes, desc="Analyzing classes"):
        class_path = os.path.join(TRAIN_DIR, class_dir)
        images = [f for f in os.listdir(class_path) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        class_counts.append(len(images))
        class_names.append(class_dir)
    
    return pd.DataFrame({
        'class': class_names,
        'image_count': class_counts
    })

# Analyze training distribution
train_df = analyze_training_distribution()

if train_df is not None:
    # Display statistics
    print(f"\n📈 Training Set Analysis (Sample of {len(train_df)} classes):")
    print(f"Average images per class: {train_df['image_count'].mean():.1f}")
    print(f"Min images per class: {train_df['image_count'].min()}")
    print(f"Max images per class: {train_df['image_count'].max()}")
    print(f"Total images (sample): {train_df['image_count'].sum():,}")
    
    # Plot distribution
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.hist(train_df['image_count'], bins=20, alpha=0.7, color='skyblue')
    plt.xlabel('Images per Class')
    plt.ylabel('Number of Classes')
    plt.title('Distribution of Images per Class\n(Training Set Sample)')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    # Top 10 classes by image count
    top_classes = train_df.nlargest(10, 'image_count')
    plt.barh(range(len(top_classes)), top_classes['image_count'])
    plt.yticks(range(len(top_classes)), 
               [c[:15] + '...' if len(c) > 15 else c for c in top_classes['class']])
    plt.xlabel('Number of Images')
    plt.title('Top 10 Classes by Image Count')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Step 6: Sample Image Exploration

def display_sample_images(num_classes=6, images_per_class=4):
    """Display sample images from different classes"""
    if not os.path.exists(TRAIN_DIR):
        print("❌ Training directory not found")
        return
    
    # Get list of classes
    train_classes = [d for d in os.listdir(TRAIN_DIR) 
                    if os.path.isdir(os.path.join(TRAIN_DIR, d))]
    
    # Randomly sample classes
    sample_classes = random.sample(train_classes, min(num_classes, len(train_classes)))
    
    fig, axes = plt.subplots(num_classes, images_per_class, 
                            figsize=(4*images_per_class, 3*num_classes))
    
    if num_classes == 1:
        axes = axes.reshape(1, -1)
    
    for i, class_dir in enumerate(sample_classes):
        class_path = os.path.join(TRAIN_DIR, class_dir)
        
        # Get image files
        image_files = [f for f in os.listdir(class_path) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        # Sample images
        sample_images = random.sample(image_files, 
                                     min(images_per_class, len(image_files)))
        
        for j, img_file in enumerate(sample_images):
            img_path = os.path.join(class_path, img_file)
            
            try:
                # Load and display image
                img = Image.open(img_path)
                
                axes[i, j].imshow(img)
                axes[i, j].axis('off')
                
                if j == 0:  # Add class name to first image
                    axes[i, j].set_title(f"{class_dir}\n{img.size[0]}x{img.size[1]}", 
                                        fontsize=10, pad=10)
                else:
                    axes[i, j].set_title(f"{img.size[0]}x{img.size[1]}", 
                                        fontsize=8)
                
            except Exception as e:
                axes[i, j].text(0.5, 0.5, f"Error loading\n{img_file}", 
                               ha='center', va='center', transform=axes[i, j].transAxes)
                axes[i, j].axis('off')
        
        # Hide unused subplots
        for j in range(len(sample_images), images_per_class):
            axes[i, j].axis('off')
    
    plt.suptitle(f'Sample Images from {num_classes} Random Classes', fontsize=16, y=0.98)
    plt.tight_layout()
    plt.show()

# Display sample images
print("🖼️ Displaying sample images from random classes...")
display_sample_images(num_classes=4, images_per_class=4)

In [ ]:
# Step 7: Analyze Image Properties

def analyze_image_properties(num_samples=200):
    """Analyze properties of sample images (size, format, etc.)"""
    if not os.path.exists(TRAIN_DIR):
        print("❌ Training directory not found")
        return
    
    image_info = []
    
    # Get list of classes
    train_classes = [d for d in os.listdir(TRAIN_DIR) 
                    if os.path.isdir(os.path.join(TRAIN_DIR, d))]
    
    # Sample images from different classes
    sample_classes = random.sample(train_classes, min(10, len(train_classes)))
    
    print(f"🔍 Analyzing properties of {num_samples} sample images...")
    
    samples_collected = 0
    
    for class_dir in sample_classes:
        if samples_collected >= num_samples:
            break
            
        class_path = os.path.join(TRAIN_DIR, class_dir)
        image_files = [f for f in os.listdir(class_path) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        # Sample images from this class
        class_samples = min(num_samples // len(sample_classes), len(image_files))
        sample_images = random.sample(image_files, class_samples)
        
        for img_file in sample_images:
            try:
                img_path = os.path.join(class_path, img_file)
                img = Image.open(img_path)
                
                # Get file size
                file_size_kb = os.path.getsize(img_path) / 1024
                
                image_info.append({
                    'class': class_dir,
                    'filename': img_file,
                    'width': img.size[0],
                    'height': img.size[1],
                    'format': img.format,
                    'mode': img.mode,
                    'file_size_kb': file_size_kb,
                    'aspect_ratio': img.size[0] / img.size[1]
                })
                
                img.close()
                samples_collected += 1
                
            except Exception as e:
                print(f"⚠️ Error processing {img_file}: {e}")
    
    if not image_info:
        print("❌ No images could be analyzed")
        return
    
    # Convert to DataFrame
    df = pd.DataFrame(image_info)
    
    # Display statistics
    print(f"\n📊 Image Properties Analysis ({len(df)} images):")
    print("=" * 50)
    print(f"Width  - Min: {df['width'].min()}, Max: {df['width'].max()}, Mean: {df['width'].mean():.1f}")
    print(f"Height - Min: {df['height'].min()}, Max: {df['height'].max()}, Mean: {df['height'].mean():.1f}")
    print(f"File Size (KB) - Min: {df['file_size_kb'].min():.1f}, Max: {df['file_size_kb'].max():.1f}, Mean: {df['file_size_kb'].mean():.1f}")
    print(f"Aspect Ratio - Min: {df['aspect_ratio'].min():.2f}, Max: {df['aspect_ratio'].max():.2f}, Mean: {df['aspect_ratio'].mean():.2f}")
    
    print(f"\nFormats: {df['format'].value_counts().to_dict()}")
    print(f"Modes: {df['mode'].value_counts().to_dict()}")
    
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Width distribution
    axes[0, 0].hist(df['width'], bins=30, alpha=0.7, color='skyblue')
    axes[0, 0].set_xlabel('Width (pixels)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Image Width Distribution')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Height distribution
    axes[0, 1].hist(df['height'], bins=30, alpha=0.7, color='lightcoral')
    axes[0, 1].set_xlabel('Height (pixels)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Image Height Distribution')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Aspect ratio distribution
    axes[1, 0].hist(df['aspect_ratio'], bins=30, alpha=0.7, color='lightgreen')
    axes[1, 0].set_xlabel('Aspect Ratio (Width/Height)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Aspect Ratio Distribution')
    axes[1, 0].grid(True, alpha=0.3)
    
    # File size distribution
    axes[1, 1].hist(df['file_size_kb'], bins=30, alpha=0.7, color='gold')
    axes[1, 1].set_xlabel('File Size (KB)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('File Size Distribution')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return df

# Analyze image properties
image_props_df = analyze_image_properties(num_samples=200)

In [ ]:
# Step 8: Explore Validation Set

def analyze_validation_set():
    """Analyze the validation set structure"""
    if not os.path.exists(VAL_DIR):
        print("❌ Validation directory not found")
        return
    
    # Get validation images
    val_images = [f for f in os.listdir(VAL_DIR) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    print(f"📊 Validation Set Analysis:")
    print(f"Total validation images: {len(val_images):,}")
    
    # Sample some validation images
    if val_images:
        sample_images = random.sample(val_images, min(8, len(val_images)))
        
        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        axes = axes.flatten()
        
        for i, img_file in enumerate(sample_images):
            try:
                img_path = os.path.join(VAL_DIR, img_file)
                img = Image.open(img_path)
                
                axes[i].imshow(img)
                axes[i].set_title(f"{img_file}\n{img.size[0]}x{img.size[1]}", fontsize=10)
                axes[i].axis('off')
                
                img.close()
                
            except Exception as e:
                axes[i].text(0.5, 0.5, f"Error loading\n{img_file}", 
                           ha='center', va='center', transform=axes[i].transAxes)
                axes[i].axis('off')
        
        plt.suptitle('Sample Validation Images', fontsize=16)
        plt.tight_layout()
        plt.show()
    
    # Check if validation ground truth file exists
    val_gt_file = os.path.join(IMAGESETS_DIR, "val.txt")
    if os.path.exists(val_gt_file):
        print(f"✅ Validation ground truth file found: {val_gt_file}")
        
        # Read first few lines
        with open(val_gt_file, 'r') as f:
            lines = f.readlines()[:10]
            print("\n📄 Sample validation ground truth entries:")
            for line in lines:
                print(f"  {line.strip()}")
    else:
        print(f"❌ Validation ground truth file not found: {val_gt_file}")

# Analyze validation set
analyze_validation_set()

In [ ]:
# Step 9: Explore Annotations (Bounding Boxes)

def parse_annotation_xml(xml_file):
    """Parse ImageNet annotation XML file"""
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        # Get image info
        filename = root.find('filename').text if root.find('filename') is not None else 'unknown'
        
        size_elem = root.find('size')
        if size_elem is not None:
            width = int(size_elem.find('width').text)
            height = int(size_elem.find('height').text)
        else:
            width, height = 0, 0
        
        # Get object annotations
        objects = []
        for obj in root.findall('object'):
            name = obj.find('name').text if obj.find('name') is not None else 'unknown'
            
            bbox = obj.find('bndbox')
            if bbox is not None:
                xmin = int(bbox.find('xmin').text)
                ymin = int(bbox.find('ymin').text)
                xmax = int(bbox.find('xmax').text)
                ymax = int(bbox.find('ymax').text)
                
                objects.append({
                    'name': name,
                    'xmin': xmin,
                    'ymin': ymin,
                    'xmax': xmax,
                    'ymax': ymax,
                    'width': xmax - xmin,
                    'height': ymax - ymin
                })
        
        return {
            'filename': filename,
            'img_width': width,
            'img_height': height,
            'objects': objects
        }
    
    except Exception as e:
        print(f"Error parsing {xml_file}: {e}")
        return None

def explore_annotations(num_samples=10):
    """Explore annotation files and visualize bounding boxes"""
    # Check training annotations
    if os.path.exists(TRAIN_ANNOTATIONS_DIR):
        print(f"✅ Training annotations directory found")
        
        # Get annotation files from first class
        ann_classes = [d for d in os.listdir(TRAIN_ANNOTATIONS_DIR) 
                      if os.path.isdir(os.path.join(TRAIN_ANNOTATIONS_DIR, d))]
        
        if ann_classes:
            sample_class = ann_classes[0]
            class_ann_dir = os.path.join(TRAIN_ANNOTATIONS_DIR, sample_class)
            
            xml_files = [f for f in os.listdir(class_ann_dir) if f.endswith('.xml')]
            print(f"📄 Found {len(xml_files)} annotation files in class '{sample_class}'")
            
            if xml_files:
                # Parse sample annotations
                sample_files = random.sample(xml_files, min(num_samples, len(xml_files)))
                
                annotations = []
                for xml_file in sample_files:
                    xml_path = os.path.join(class_ann_dir, xml_file)
                    ann = parse_annotation_xml(xml_path)
                    if ann:
                        annotations.append(ann)
                
                if annotations:
                    print(f"\n📊 Annotation Analysis (Sample of {len(annotations)} files):")
                    
                    # Analyze object counts
                    obj_counts = [len(ann['objects']) for ann in annotations]
                    print(f"Objects per image - Min: {min(obj_counts)}, Max: {max(obj_counts)}, Mean: {np.mean(obj_counts):.1f}")
                    
                    # Visualize some images with bounding boxes
                    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
                    axes = axes.flatten()
                    
                    for i, ann in enumerate(annotations[:6]):
                        # Try to load corresponding image
                        img_name = ann['filename']
                        img_path = os.path.join(TRAIN_DIR, sample_class, img_name)
                        
                        if os.path.exists(img_path):
                            try:
                                img = Image.open(img_path)
                                axes[i].imshow(img)
                                
                                # Draw bounding boxes
                                for obj in ann['objects']:
                                    rect = patches.Rectangle(
                                        (obj['xmin'], obj['ymin']),
                                        obj['width'], obj['height'],
                                        linewidth=2, edgecolor='red', facecolor='none'
                                    )
                                    axes[i].add_patch(rect)
                                    
                                    # Add label
                                    axes[i].text(obj['xmin'], obj['ymin']-5, obj['name'], 
                                               color='red', fontsize=8, weight='bold')
                                
                                axes[i].set_title(f"{img_name}\n{len(ann['objects'])} objects", fontsize=10)
                                axes[i].axis('off')
                                
                                img.close()
                                
                            except Exception as e:
                                axes[i].text(0.5, 0.5, f"Error loading\n{img_name}", 
                                           ha='center', va='center', transform=axes[i].transAxes)
                                axes[i].axis('off')
                        else:
                            axes[i].text(0.5, 0.5, f"Image not found\n{img_name}", 
                                       ha='center', va='center', transform=axes[i].transAxes)
                            axes[i].axis('off')
                    
                    plt.suptitle(f'Sample Images with Bounding Box Annotations\nClass: {sample_class}', fontsize=16)
                    plt.tight_layout()
                    plt.show()
    
    else:
        print(f"❌ Training annotations directory not found: {TRAIN_ANNOTATIONS_DIR}")

# Explore annotations
print("🔍 Exploring annotation files...")
explore_annotations(num_samples=6)

In [ ]:
# Step 10: Dataset Summary and Next Steps

def print_dataset_summary():
    """Print a comprehensive summary of the dataset exploration"""
    print("🎯 ImageNet Dataset Exploration Summary")
    print("=" * 60)
    
    # Check what we found
    found_components = []
    missing_components = []
    
    components = {
        "Training Images": TRAIN_DIR,
        "Validation Images": VAL_DIR,
        "Test Images": TEST_DIR,
        "Training Annotations": TRAIN_ANNOTATIONS_DIR,
        "Validation Annotations": VAL_ANNOTATIONS_DIR,
        "ImageSets": IMAGESETS_DIR
    }
    
    for name, path in components.items():
        if os.path.exists(path):
            found_components.append(name)
        else:
            missing_components.append(name)
    
    print(f"✅ Found Components ({len(found_components)}):")
    for component in found_components:
        print(f"   • {component}")
    
    if missing_components:
        print(f"\n❌ Missing Components ({len(missing_components)}):")
        for component in missing_components:
            print(f"   • {component}")
    
    print(f"\n📊 Quick Stats:")
    if os.path.exists(TRAIN_DIR):
        train_classes = [d for d in os.listdir(TRAIN_DIR) 
                        if os.path.isdir(os.path.join(TRAIN_DIR, d))]
        print(f"   • Training classes: {len(train_classes)}")
    
    if os.path.exists(VAL_DIR):
        val_images = [f for f in os.listdir(VAL_DIR) 
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"   • Validation images: {len(val_images):,}")
    
    if os.path.exists(TEST_DIR):
        test_images = [f for f in os.listdir(TEST_DIR) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"   • Test images: {len(test_images):,}")
    
    print(f"\n🎯 Dataset is ready for:")
    print(f"   • Image classification tasks")
    if os.path.exists(TRAIN_ANNOTATIONS_DIR):
        print(f"   • Object detection/localization tasks")
    print(f"   • Transfer learning experiments")
    print(f"   • Computer vision research")
    
    print(f"\n📚 Next Steps:")
    print(f"   1. Load specific classes for your task")
    print(f"   2. Implement data loaders (PyTorch/TensorFlow)")
    print(f"   3. Apply data augmentation techniques")
    print(f"   4. Train/fine-tune deep learning models")
    print(f"   5. Evaluate on validation/test sets")

# Print summary
print_dataset_summary()

## 🚀 Usage Examples

### Loading Data with PyTorch
```python
import torch
from torchvision import datasets, transforms

# Define transforms
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                        std=[0.229, 0.224, 0.225])
])

# Load dataset
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
```

### Custom Dataset Class
```python
class ImageNetDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        # Implement your custom loading logic
        
    def __len__(self):
        return len(self.image_paths)
        
    def __getitem__(self, idx):
        # Load image and apply transforms
        return image, label
```

---

**Dataset Path**: `/home/ubuntu/Downloads/ILSVRC`  
**Notebook**: ImageNet Dataset Explorer  
**Last Updated**: October 2025